In [1]:
from pathlib import Path
import pandas as pd

# ============================================================
# Configuración base
# ============================================================

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

FILES = {
    "01_candidatos_normalizados": ROOT / "data/processed/met_candidatos_normalizados.csv",
    "02_aceptados_filtro_textil": ROOT / "data/processed/met_textiles_pilot_clean.csv",
    "03_rechazados_filtro_textil": ROOT / "data/processed/met_textiles_rejected.csv",
    "04_corpus_piloto": ROOT / "data/metadata/corpus_piloto.csv",
    "05_corpus_piloto_clean": ROOT / "data/metadata/corpus_piloto_clean.csv",
    "06_corpus_piloto_secondary": ROOT / "data/metadata/corpus_piloto_secondary.csv",
    "07_corpus_piloto_duplicates": ROOT / "data/metadata/corpus_piloto_duplicates.csv",
}

def read_csv(path):
    if not path.exists():
        print(f"No existe: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

dfs = {name: read_csv(path) for name, path in FILES.items()}

resumen = []

for name, df in dfs.items():
    resumen.append({
        "archivo": name,
        "filas": len(df),
        "columnas": len(df.columns),
        "tiene_id_objeto": "id_objeto" in df.columns,
        "tiene_url_imagen": any(c in df.columns for c in ["url_imagen", "primaryImage", "primaryImageSmall"]),
        "columnas_motivo": [c for c in df.columns if "motivo" in c.lower()],
    })

pd.DataFrame(resumen)

,archivo,filas,columnas,tiene_id_objeto,tiene_url_imagen,columnas_motivo
0,01_candidatos_normalizados,295,33,True,True,[motivo_curacion]
1,02_aceptados_filtro_textil,211,39,True,True,"[motivo_curacion, motivo_filtrado]"
2,03_rechazados_filtro_textil,84,39,True,True,"[motivo_curacion, motivo_filtrado]"
3,04_corpus_piloto,50,40,True,False,"[motivo_superficie, motivos, motivo_filtrado]"
4,05_corpus_piloto_clean,35,41,True,False,"[motivo_superficie, motivos, motivo_filtrado]"
5,06_corpus_piloto_secondary,15,42,True,False,"[motivo_superficie, motivos, motivo_filtrado, ..."
6,07_corpus_piloto_duplicates,0,41,True,False,"[motivo_superficie, motivos, motivo_filtrado, ..."


In [2]:
# ============================================================
# Auditoría de pasos del pipeline
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x

def get_ids(df):
    if df.empty or "id_objeto" not in df.columns:
        return set()
    return set(df["id_objeto"].apply(norm_id))

ids = {name: get_ids(df) for name, df in dfs.items()}

transiciones = [
    ("01_candidatos_normalizados", "02_aceptados_filtro_textil"),
    ("01_candidatos_normalizados", "03_rechazados_filtro_textil"),
    ("02_aceptados_filtro_textil", "04_corpus_piloto"),
    ("04_corpus_piloto", "05_corpus_piloto_clean"),
    ("04_corpus_piloto", "06_corpus_piloto_secondary"),
    ("04_corpus_piloto", "07_corpus_piloto_duplicates"),
]

rows = []

for origen, destino in transiciones:
    origen_ids = ids[origen]
    destino_ids = ids[destino]
    
    rows.append({
        "origen": origen,
        "destino": destino,
        "ids_origen": len(origen_ids),
        "ids_destino": len(destino_ids),
        "ids_en_comun": len(origen_ids & destino_ids),
        "ids_que_no_pasaron": len(origen_ids - destino_ids),
    })

pd.DataFrame(rows)

,origen,destino,ids_origen,ids_destino,ids_en_comun,ids_que_no_pasaron
0,01_candidatos_normalizados,02_aceptados_filtro_textil,295,211,211,84
1,01_candidatos_normalizados,03_rechazados_filtro_textil,295,84,84,211
2,02_aceptados_filtro_textil,04_corpus_piloto,211,50,50,161
3,04_corpus_piloto,05_corpus_piloto_clean,50,35,35,15
4,04_corpus_piloto,06_corpus_piloto_secondary,50,15,15,35
5,04_corpus_piloto,07_corpus_piloto_duplicates,50,0,0,50


In [3]:
# ============================================================
# Distribución de motivos por archivo
# ============================================================

for name, df in dfs.items():
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    motivo_cols = [c for c in df.columns if "motivo" in c.lower()]
    
    if not motivo_cols:
        print("No tiene columnas de motivo.")
        continue
    
    for col in motivo_cols:
        print(f"\nColumna: {col}")
        display(df[col].fillna("(vacío)").value_counts().reset_index().rename(
            columns={"index": col, col: "conteo"}
        ))


01_candidatos_normalizados

Columna: motivo_curacion


,conteo,count
0,clasificación contiene evidencia textil: ['tex...,20
1,clasificación contiene evidencia textil: ['tex...,18
2,clasificación contiene evidencia textil: ['tex...,12
3,clasificación contiene evidencia textil: ['tex...,11
4,título contiene términos textiles: ['textile']...,9
...,...,...
139,clasificación contiene evidencia textil: ['tex...,1
140,clasificación contiene evidencia textil: ['tex...,1
141,clasificación contiene evidencia textil: ['tex...,1
142,clasificación contiene evidencia textil: ['tex...,1



02_aceptados_filtro_textil

Columna: motivo_curacion


,conteo,count
0,clasificación contiene evidencia textil: ['tex...,20
1,clasificación contiene evidencia textil: ['tex...,18
2,clasificación contiene evidencia textil: ['tex...,12
3,clasificación contiene evidencia textil: ['tex...,11
4,clasificación contiene evidencia textil: ['tex...,8
...,...,...
93,clasificación contiene evidencia textil: ['tex...,1
94,clasificación contiene evidencia textil: ['tex...,1
95,clasificación contiene evidencia textil: ['tex...,1
96,clasificación contiene evidencia textil: ['tex...,1



Columna: motivo_filtrado


...

,conteo,count



Columna: motivo_filtrado


,conteo,count



Columna: motivo_duplicado


,conteo,count


In [4]:
# ============================================================
# Identificar registros aceptados que NO pasaron al corpus piloto
# ============================================================

aceptados = dfs["02_aceptados_filtro_textil"].copy()
corpus_piloto = dfs["04_corpus_piloto"].copy()

ids_piloto = get_ids(corpus_piloto)

aceptados_no_piloto = aceptados[
    ~aceptados["id_objeto"].apply(norm_id).isin(ids_piloto)
].copy()

print("Aceptados por filtro textil:", len(aceptados))
print("Pasaron a corpus piloto:", len(corpus_piloto))
print("Aceptados que NO pasaron a corpus piloto:", len(aceptados_no_piloto))

cols_revision = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "motivo_filtrado",
    "url_objeto",
    "url_imagen",
    "primaryImage",
    "primaryImageSmall",
]

cols_revision = [c for c in cols_revision if c in aceptados_no_piloto.columns]

aceptados_no_piloto[cols_revision].head(20)

Aceptados por filtro textil: 211
Pasaron a corpus piloto: 50
Aceptados que NO pasaron a corpus piloto: 161


,id_objeto,titulo_original,cultura,periodo,clasificacion_original,nombre_objeto_original,material_original,motivo_filtrado,url_objeto,url_imagen
11,316983,Four-Cornered Hat,Tiwanaku,NaN,Textiles-Costumes,Hat,Camelid hair,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
12,316981,Four-Cornered Hat,Tiwanaku,NaN,Textiles-Costumes,Hat,Camelid hair,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
26,315772,Headband with ornamental tassels,Nasca,NaN,Textiles-Woven,Tassels,Camelid hair,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
28,313148,Weaving Basket,Chancay,NaN,Textiles-Implements,Basket with weaving implements,"Cane, shell, bone, fiber, camelid hair, wood, ...",clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
29,314894,Square Cloth with Feathered Border,Chimú,NaN,Textiles-Featherwork,Panel,"Feathers, camelid hair, cotton, wool",clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
33,308078,"Border fragment, figures with staves",Paracas,NaN,Textiles-Woven,Fragment,Camelid hair,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
35,307944,Sash,Chimú,NaN,Textiles-Woven,Sash,Camelid hair,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
40,308131,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
41,316948,Openweave Cap,Peruvian,NaN,Textiles-Costumes,Hat,Cotton,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
42,316910,Sash Fragment,Peruvian,NaN,Textiles-Woven,Sash fragment,Cotton,clasificación contiene evidencia textil: ['tex...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...


In [5]:
# ============================================================
# Resumen de los aceptados que no pasaron al corpus piloto
# ============================================================

for col in [
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "cultura",
]:
    if col in aceptados_no_piloto.columns:
        print("\n" + "=" * 80)
        print(col)
        print("=" * 80)
        display(
            aceptados_no_piloto[col]
            .fillna("(vacío)")
            .value_counts()
            .head(30)
            .reset_index()
            .rename(columns={"index": col, col: "conteo"})
        )


clasificacion_original


,conteo,count
0,Textiles-Woven,111
1,Textiles-Costumes,27
2,Textiles,11
3,Textiles-Non-Woven,5
4,Textiles-Featherwork,4
5,Textiles-Implements,1
6,Feathers-Costumes,1
7,Textiles-Velvets,1



nombre_objeto_original


,conteo,count
0,Hat,29
1,Textile fragment,20
2,Mantle fragment,19
3,Bag,13
4,Skein,11
5,Border Fragment,8
6,Panel,7
7,Headband,7
8,Band fragment,6
9,Sash,5



material_original


,conteo,count
0,Camelid hair,117
1,Cotton,11
2,"Camelid hair, cotton",11
3,"Cotton, camelid hair",6
4,"Cotton, paint",5
5,"Cotton, feathers",2
6,"Cotton, pigment",2
7,Camelid fiber,2
8,"Cane, shell, bone, fiber, camelid hair, wood, ...",1
9,"Feathers, camelid hair, cotton, wool",1



cultura


,conteo,count
0,Nasca,40
1,Paracas,39
2,Peruvian,24
3,Wari,21
4,Chancay,14
5,Inca,10
6,Nasca (?),5
7,Chimú,3
8,Tiwanaku,2
9,Inca (?),1


In [6]:
# ============================================================
# Crear galería visual de aceptados que NO pasaron a corpus piloto
# ============================================================

from pathlib import Path
import html

OUTPUT_DIR = ROOT / "outputs/review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_image_col(df):
    candidates = [
        "url_imagen",
        "image_url",
        "imagen_url",
        "primaryImageSmall",
        "primaryImage",
        "url_imagen_principal",
    ]
    for col in candidates:
        if col in df.columns:
            return col
    return None

def find_object_url_col(df):
    candidates = ["url_objeto", "objectURL", "url", "link"]
    for col in candidates:
        if col in df.columns:
            return col
    return None

image_col = find_image_col(aceptados_no_piloto)
object_url_col = find_object_url_col(aceptados_no_piloto)

print("Columna de imagen detectada:", image_col)
print("Columna de URL objeto detectada:", object_url_col)

def safe_value(row, col):
    if col is None or col not in row.index:
        return ""
    value = row[col]
    if pd.isna(value):
        return ""
    return str(value)

def build_gallery(df, output_path, title):
    cards = []

    for _, row in df.iterrows():
        img = safe_value(row, image_col)
        obj_url = safe_value(row, object_url_col)

        object_id = safe_value(row, "id_objeto")
        titulo = safe_value(row, "titulo_original")
        cultura = safe_value(row, "cultura")
        periodo = safe_value(row, "periodo")
        clasificacion = safe_value(row, "clasificacion_original")
        nombre_objeto = safe_value(row, "nombre_objeto_original")
        material = safe_value(row, "material_original")
        motivo = safe_value(row, "motivo_filtrado")

        link_html = ""
        if obj_url:
            link_html = f'<p><a href="{html.escape(obj_url)}" target="_blank">Ver objeto en museo</a></p>'

        img_html = ""
        if img:
            img_html = f'<img src="{html.escape(img)}" loading="lazy">'
        else:
            img_html = '<div class="no-image">Sin imagen detectada</div>'

        card = f"""
        <div class="card">
            {img_html}
            <div class="info">
                <h3>{html.escape(object_id)} - {html.escape(titulo)}</h3>
                <p><b>Cultura:</b> {html.escape(cultura)}</p>
                <p><b>Periodo:</b> {html.escape(periodo)}</p>
                <p><b>Clasificación:</b> {html.escape(clasificacion)}</p>
                <p><b>Objeto:</b> {html.escape(nombre_objeto)}</p>
                <p><b>Material:</b> {html.escape(material)}</p>
                <p><b>Motivo actual:</b> {html.escape(motivo)}</p>
                {link_html}
                <p><b>Revisión manual:</b> ______________________________</p>
                <p><b>Decisión sugerida:</b> [ ] pasa a piloto | [ ] secundario | [ ] descartar</p>
            </div>
        </div>
        """
        cards.append(card)

    html_text = f"""
    <!doctype html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>{html.escape(title)}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 24px;
                background: #f6f6f6;
            }}
            h1 {{
                margin-bottom: 8px;
            }}
            .summary {{
                margin-bottom: 24px;
                color: #333;
            }}
            .grid {{
                display: grid;
                grid-template-columns: repeat(auto-fill, minmax(320px, 1fr));
                gap: 18px;
            }}
            .card {{
                background: white;
                border: 1px solid #ddd;
                border-radius: 8px;
                overflow: hidden;
            }}
            img {{
                width: 100%;
                height: 280px;
                object-fit: contain;
                background: #eee;
                border-bottom: 1px solid #ddd;
            }}
            .no-image {{
                height: 280px;
                display: flex;
                align-items: center;
                justify-content: center;
                background: #ddd;
                color: #555;
            }}
            .info {{
                padding: 12px;
                font-size: 14px;
            }}
            h3 {{
                font-size: 15px;
                margin-top: 0;
            }}
            p {{
                margin: 6px 0;
            }}
        </style>
    </head>
    <body>
        <h1>{html.escape(title)}</h1>
        <p class="summary">Total registros: {len(df)}</p>
        <div class="grid">
            {''.join(cards)}
        </div>
    </body>
    </html>
    """

    output_path.write_text(html_text, encoding="utf-8")
    return output_path

gallery_path = build_gallery(
    aceptados_no_piloto,
    OUTPUT_DIR / "auditoria_aceptados_no_piloto.html",
    "Auditoría visual - Aceptados que no pasaron a corpus piloto"
)

gallery_path

Columna de imagen detectada: url_imagen
Columna de URL objeto detectada: url_objeto


WindowsPath('d:/Documentos/000. MSC/00. Tesis-Textiles/uni-cc-base-multimodal-textiles-andinos/outputs/review/auditoria_aceptados_no_piloto.html')

In [7]:
gallery_path

WindowsPath('d:/Documentos/000. MSC/00. Tesis-Textiles/uni-cc-base-multimodal-textiles-andinos/outputs/review/auditoria_aceptados_no_piloto.html')

In [8]:
# ============================================================
# Clasificación preliminar para auditar los 161 aceptados no piloto
# ============================================================

def text_norm(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()

def clasificar_revision(row):
    nombre = text_norm(row.get("nombre_objeto_original", ""))
    titulo = text_norm(row.get("titulo_original", ""))
    clasif = text_norm(row.get("clasificacion_original", ""))
    material = text_norm(row.get("material_original", ""))

    texto = " ".join([nombre, titulo, clasif, material])

    # Materiales o implementos textiles, pero no superficie visual
    if "skein" in texto:
        return "descartar_material_sin_superficie", "madeja/hilo sin superficie visual útil"

    if "basket" in texto or "implement" in texto:
        return "descartar_implemento_textil", "implemento asociado al tejido, no textil analizable"

    # Featherwork: importante, pero conviene separarlo del corpus principal
    if "feather" in texto:
        return "secundario_featherwork", "textil con plumas / featherwork, revisar como corpus secundario"

    # Objetos estrechos o longitudinales
    if any(k in texto for k in ["sash", "headband", "band", "belt"]):
        return "revision_formato_longitudinal", "formato longitudinal; revisar si contiene motivos útiles"

    # Prendas pequeñas o tridimensionales
    if any(k in texto for k in ["hat", "cap", "mask", "headdress"]):
        return "revision_prenda_accesorio", "prenda/accesorio; revisar si aporta iconografía o patrón"

    # Posibles piezas principales para el corpus
    if any(k in texto for k in [
        "textile fragment",
        "mantle fragment",
        "mantle",
        "bag",
        "panel",
        "tunic",
        "tabard",
        "hanging",
        "fragment",
        "border"
    ]):
        return "candidato_corpus_principal", "posible superficie textil útil; requiere revisión visual"

    return "revision_manual", "caso no cubierto por reglas preliminares"

aceptados_no_piloto[["categoria_revision", "motivo_revision_preliminar"]] = aceptados_no_piloto.apply(
    lambda row: pd.Series(clasificar_revision(row)),
    axis=1
)

display(
    aceptados_no_piloto["categoria_revision"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "categoria_revision", "categoria_revision": "conteo"})
)

,conteo,count
0,candidato_corpus_principal,78
1,revision_formato_longitudinal,32
2,revision_prenda_accesorio,29
3,descartar_material_sin_superficie,11
4,secundario_featherwork,5
5,revision_manual,5
6,descartar_implemento_textil,1


In [9]:
# ============================================================
# Ver ejemplos por categoría preliminar
# ============================================================

cols = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "categoria_revision",
    "motivo_revision_preliminar",
    "url_objeto",
    "url_imagen",
]

cols = [c for c in cols if c in aceptados_no_piloto.columns]

for categoria in aceptados_no_piloto["categoria_revision"].unique():
    print("\n" + "=" * 100)
    print(categoria)
    print("=" * 100)
    display(
        aceptados_no_piloto.loc[
            aceptados_no_piloto["categoria_revision"] == categoria,
            cols
        ].head(15)
    )


revision_prenda_accesorio


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision,motivo_revision_preliminar,url_objeto,url_imagen
11,316983,Four-Cornered Hat,Tiwanaku,Textiles-Costumes,Hat,Camelid hair,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
12,316981,Four-Cornered Hat,Tiwanaku,Textiles-Costumes,Hat,Camelid hair,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
41,316948,Openweave Cap,Peruvian,Textiles-Costumes,Hat,Cotton,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
43,316949,Openweave Cap with Tassels,Peruvian,Textiles-Costumes,Hat,Cotton,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
57,315785,Hat,Peruvian,Textiles-Woven,Hat,Cotton,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
61,316940,Hat,Peruvian,Textiles-Costumes,Hat,Camelid hair,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
62,316955,Cap,Peruvian,Textiles-Costumes,Hat,Camelid hair,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
93,314614,Cap Woven with Human Hair,Inca,Textiles-Woven,Hat,"Camelid hair, human hair",revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
130,318460,False Face for Funerary Bundle,Paracas,Textiles-Woven,Mask,"Cotton, paint",revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
176,316342,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_accesorio,prenda/accesorio; revisar si aporta iconografí...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



revision_formato_longitudinal


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision,motivo_revision_preliminar,url_objeto,url_imagen
26,315772,Headband with ornamental tassels,Nasca,Textiles-Woven,Tassels,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
35,307944,Sash,Chimú,Textiles-Woven,Sash,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
42,316910,Sash Fragment,Peruvian,Textiles-Woven,Sash fragment,Cotton,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
55,316916,Sash Fragments,Peruvian,Textiles-Woven,Sash fragment,Cotton,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
56,316915,Sash Fragments,Peruvian,Textiles-Woven,Sash fragment,Cotton,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
63,316931,Headband,Peruvian,Textiles-Woven,Headband,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
64,316894,Sash Fragment with Fringe,Peruvian,Textiles-Woven,Sash fragment,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
76,309528,Band,Inca,Textiles-Woven,Band,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
82,318888,Headband,Inca,Textiles-Woven,Headband,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
86,314528,Tunic with Diamond Band,Inca,Textiles-Woven,Tunic,"Camelid hair, cotton",revision_formato_longitudinal,formato longitudinal; revisar si contiene moti...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



descartar_implemento_textil


...

,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision,motivo_revision_preliminar,url_objeto,url_imagen
33,308078,"Border fragment, figures with staves",Paracas,Textiles-Woven,Fragment,Camelid hair,candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
60,308084,Tapestry Border Fragment,Peruvian,Textiles-Woven,Border Fragment,Camelid hair,candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
68,316945,Border Fragment,Paracas,Textiles-Woven,Border fragment,"Cotton, camelid hair",candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
92,313324,Sleeveless Tunic,Inca,Textiles-Woven,Tunic,"Cotton, camelid hair",candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
94,308046,Embroidered Border Fragment,Paracas,Textiles-Velvets,Border Fragment,Camelid hair,candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
95,310511,Mantle Fragment,Paracas,Textiles-Woven,Mantle fragment,"Cotton, pigment",candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
96,307853,Embroidered Fragment,Paracas,Textiles-Woven,Textile fragment,Camelid hair,candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
97,308056,Embroidered Fragment,Paracas,Textiles-Woven,Textile fragment,Camelid hair,candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
98,308106,Embroidered Fragment,Paracas,Textiles-Woven,Textile fragment,Camelid hair,candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
99,308100,Embroidered Fragment,Paracas,Textiles-Woven,Textile fragment,Camelid hair,candidato_corpus_principal,posible superficie textil útil; requiere revis...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



descartar_material_sin_superficie


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision,motivo_revision_preliminar,url_objeto,url_imagen
40,308131,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
44,308125,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
45,308126,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
46,308127,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
47,308128,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
48,308124,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
49,308130,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
50,308133,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
51,308132,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
52,308134,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,madeja/hilo sin superficie visual útil,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



revision_manual


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision,motivo_revision_preliminar,url_objeto,url_imagen
78,314616,Woven Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,revision_manual,caso no cubierto por reglas preliminares,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
79,316928,Sling,Inca (?),Textiles-Woven,Sling,Camelid hair,revision_manual,caso no cubierto por reglas preliminares,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
80,314617,Woven Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,revision_manual,caso no cubierto por reglas preliminares,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
81,314615,Woven Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,revision_manual,caso no cubierto por reglas preliminares,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
85,315773,Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,revision_manual,caso no cubierto por reglas preliminares,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...


In [10]:
# ============================================================
# Lote 1: posibles recuperables para corpus principal
# ============================================================

lote_01 = aceptados_no_piloto[
    aceptados_no_piloto["categoria_revision"] == "candidato_corpus_principal"
].copy()

print("Registros en lote 01:", len(lote_01))

gallery_path = build_gallery(
    lote_01,
    OUTPUT_DIR / "auditoria_lote_01_candidatos_corpus_principal.html",
    "Lote 01 - Posibles recuperables para corpus principal"
)

gallery_path

Registros en lote 01: 78


WindowsPath('d:/Documentos/000. MSC/00. Tesis-Textiles/uni-cc-base-multimodal-textiles-andinos/outputs/review/auditoria_lote_01_candidatos_corpus_principal.html')

In [13]:
# ============================================================
# Clasificación preliminar mejorada para auditoría
# ============================================================

def text_norm(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()

def contiene(texto, palabras):
    return any(p in texto for p in palabras)

def clasificar_revision_v2(row):
    nombre = text_norm(row.get("nombre_objeto_original", ""))
    titulo = text_norm(row.get("titulo_original", ""))
    clasif = text_norm(row.get("clasificacion_original", ""))
    material = text_norm(row.get("material_original", ""))
    cultura = text_norm(row.get("cultura", ""))

    texto = " ".join([nombre, titulo, clasif, material, cultura])

    # 1. Casos sin superficie textil analizable
    if "skein" in nombre or "skein" in titulo:
        return "descartar_material_sin_superficie", "madeja/hilo: material textil, pero sin superficie visual para análisis"

    if "basket" in nombre or "implement" in clasif or "implement" in nombre:
        return "descartar_implemento_textil", "implemento asociado al tejido, no pieza textil analizable"

    # 2. Featherwork: textil válido, pero corpus secundario por materialidad especial
    if "feather" in texto:
        return "secundario_featherwork", "textil con plumas; conservar en corpus secundario"

    # 3. Piezas principales: prioridad alta
    # Esto corrige casos como 'Tunic with Diamond Band'
    if contiene(nombre, ["tunic", "mantle", "panel", "bag", "tabard", "hanging"]):
        return "candidato_corpus_principal", "pieza textil con superficie o composición potencialmente útil"

    if contiene(titulo, ["tunic", "mantle", "panel", "bag", "tabard", "hanging"]):
        return "candidato_corpus_principal", "título indica pieza textil principal con superficie útil"

    # 4. Fragmentos textiles: revisar visualmente, muchos pueden recuperarse
    if contiene(nombre, ["textile fragment", "mantle fragment", "tunic fragment", "bag fragment", "fragment"]):
        return "candidato_corpus_principal", "fragmento textil potencialmente útil; requiere revisión visual"

    # 5. Bordes y bandas: no descartarlos automáticamente
    if contiene(nombre, ["border fragment", "border fragments"]) or contiene(titulo, ["border fragment", "border fragments"]):
        return "revision_borde_iconografico", "borde/fragmento con posible iconografía; revisar legibilidad visual"

    # 6. Formatos longitudinales
    if contiene(nombre, ["sash", "headband", "band", "belt"]) or contiene(titulo, ["sash", "headband", "band fragment", "belt"]):
        return "revision_formato_longitudinal", "formato longitudinal; revisar si aporta motivos útiles"

    # 7. Prendas/accesorios con posible valor iconográfico
    if contiene(nombre, ["hat", "cap", "mask", "headdress"]) or contiene(titulo, ["hat", "cap", "mask", "headdress"]):
        if contiene(cultura, ["wari", "tiwanaku"]):
            return "revision_prenda_iconografica_prioritaria", "prenda Wari/Tiwanaku con posible valor iconográfico"
        return "revision_prenda_accesorio", "prenda/accesorio textil; revisar si aporta patrón o iconografía"

    # 8. Hondas y artefactos textiles
    if contiene(nombre, ["sling", "sling shot"]) or contiene(titulo, ["sling", "sling shot"]):
        return "secundario_artefacto_textil", "artefacto textil funcional; conservar como secundario o revisar aparte"

    return "revision_manual", "caso no cubierto por reglas preliminares"

aceptados_no_piloto[["categoria_revision_v2", "motivo_revision_v2"]] = aceptados_no_piloto.apply(
    lambda row: pd.Series(clasificar_revision_v2(row)),
    axis=1
)

display(
    aceptados_no_piloto["categoria_revision_v2"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "categoria_revision_v2", "categoria_revision_v2": "conteo"})
)

,conteo,count
0,candidato_corpus_principal,94
1,revision_prenda_iconografica_prioritaria,22
2,revision_formato_longitudinal,16
3,descartar_material_sin_superficie,11
4,revision_prenda_accesorio,7
5,secundario_featherwork,5
6,secundario_artefacto_textil,5
7,descartar_implemento_textil,1


In [14]:
# ============================================================
# Ejemplos por categoría mejorada
# ============================================================

cols = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "categoria_revision_v2",
    "motivo_revision_v2",
    "url_objeto",
    "url_imagen",
]

cols = [c for c in cols if c in aceptados_no_piloto.columns]

for categoria in aceptados_no_piloto["categoria_revision_v2"].unique():
    print("\n" + "=" * 100)
    print(categoria)
    print("=" * 100)
    display(
        aceptados_no_piloto.loc[
            aceptados_no_piloto["categoria_revision_v2"] == categoria,
            cols
        ].head(20)
    )


revision_prenda_iconografica_prioritaria


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision_v2,motivo_revision_v2,url_objeto,url_imagen
11,316983,Four-Cornered Hat,Tiwanaku,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
12,316981,Four-Cornered Hat,Tiwanaku,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
176,316342,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
177,316727,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
178,314620,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
179,316968,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
180,316972,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
181,316975,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
182,316964,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
183,316977,Four-Cornered Hat,Wari,Textiles-Costumes,Hat,Camelid hair,revision_prenda_iconografica_prioritaria,prenda Wari/Tiwanaku con posible valor iconogr...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



revision_formato_longitudinal


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision_v2,motivo_revision_v2,url_objeto,url_imagen
26,315772,Headband with ornamental tassels,Nasca,Textiles-Woven,Tassels,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
35,307944,Sash,Chimú,Textiles-Woven,Sash,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
63,316931,Headband,Peruvian,Textiles-Woven,Headband,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
76,309528,Band,Inca,Textiles-Woven,Band,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
82,318888,Headband,Inca,Textiles-Woven,Headband,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
131,312933,Headband,Nasca,Textiles-Woven,Headdress,Camelid fiber,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
146,308018,Sash Fragment,Nasca,Textiles-Woven,Sash,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
147,308020,Sash Fragment,Nasca,Textiles-Woven,Sash,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
150,308019,Sash Fragment,Nasca,Textiles-Woven,Sash,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
159,307959,Sash or Headband,Nasca,Textiles-Woven,Sash,Camelid hair,revision_formato_longitudinal,formato longitudinal; revisar si aporta motivo...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



descartar_implemento_textil


...

,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision_v2,motivo_revision_v2,url_objeto,url_imagen
40,308131,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
44,308125,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
45,308126,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
46,308127,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
47,308128,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
48,308124,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
49,308130,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
50,308133,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
51,308132,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
52,308134,Skein of Camelid Hair,Peruvian,Textiles,Skein,Camelid hair,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



revision_prenda_accesorio


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision_v2,motivo_revision_v2,url_objeto,url_imagen
41,316948,Openweave Cap,Peruvian,Textiles-Costumes,Hat,Cotton,revision_prenda_accesorio,prenda/accesorio textil; revisar si aporta pat...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
43,316949,Openweave Cap with Tassels,Peruvian,Textiles-Costumes,Hat,Cotton,revision_prenda_accesorio,prenda/accesorio textil; revisar si aporta pat...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
57,315785,Hat,Peruvian,Textiles-Woven,Hat,Cotton,revision_prenda_accesorio,prenda/accesorio textil; revisar si aporta pat...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
61,316940,Hat,Peruvian,Textiles-Costumes,Hat,Camelid hair,revision_prenda_accesorio,prenda/accesorio textil; revisar si aporta pat...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
62,316955,Cap,Peruvian,Textiles-Costumes,Hat,Camelid hair,revision_prenda_accesorio,prenda/accesorio textil; revisar si aporta pat...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
93,314614,Cap Woven with Human Hair,Inca,Textiles-Woven,Hat,"Camelid hair, human hair",revision_prenda_accesorio,prenda/accesorio textil; revisar si aporta pat...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
130,318460,False Face for Funerary Bundle,Paracas,Textiles-Woven,Mask,"Cotton, paint",revision_prenda_accesorio,prenda/accesorio textil; revisar si aporta pat...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...



secundario_artefacto_textil


,id_objeto,titulo_original,cultura,clasificacion_original,nombre_objeto_original,material_original,categoria_revision_v2,motivo_revision_v2,url_objeto,url_imagen
78,314616,Woven Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,secundario_artefacto_textil,artefacto textil funcional; conservar como sec...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
79,316928,Sling,Inca (?),Textiles-Woven,Sling,Camelid hair,secundario_artefacto_textil,artefacto textil funcional; conservar como sec...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
80,314617,Woven Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,secundario_artefacto_textil,artefacto textil funcional; conservar como sec...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
81,314615,Woven Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,secundario_artefacto_textil,artefacto textil funcional; conservar como sec...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...
85,315773,Sling Shot,Inca,Textiles-Woven,Sling shot,Camelid hair,secundario_artefacto_textil,artefacto textil funcional; conservar como sec...,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...


In [15]:
# ============================================================
# Generar galerías por categoría mejorada
# ============================================================

categorias_a_revisar = [
    "candidato_corpus_principal",
    "revision_borde_iconografico",
    "revision_prenda_iconografica_prioritaria",
    "revision_formato_longitudinal",
    "revision_prenda_accesorio",
    "secundario_featherwork",
    "secundario_artefacto_textil",
]

for categoria in categorias_a_revisar:
    lote = aceptados_no_piloto[
        aceptados_no_piloto["categoria_revision_v2"] == categoria
    ].copy()

    if len(lote) == 0:
        continue

    path = build_gallery(
        lote,
        OUTPUT_DIR / f"auditoria_{categoria}.html",
        f"Auditoría visual - {categoria}"
    )

    print(categoria, len(lote), path)

candidato_corpus_principal 94 d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_candidato_corpus_principal.html
revision_prenda_iconografica_prioritaria 22 d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_revision_prenda_iconografica_prioritaria.html
revision_formato_longitudinal 16 d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_revision_formato_longitudinal.html
revision_prenda_accesorio 7 d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_revision_prenda_accesorio.html
secundario_featherwork 5 d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_secundario_featherwork.html
secundario_artefacto_textil 5 d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\audito

In [16]:
# ============================================================
# Crear tabla de auditoría manual para los 161 aceptados no piloto
# ============================================================

audit_cols = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "url_objeto",
    "url_imagen",
    "motivo_filtrado",
    "categoria_revision_v2",
    "motivo_revision_v2",
]

audit_cols = [c for c in audit_cols if c in aceptados_no_piloto.columns]

auditoria_aceptados_no_piloto = aceptados_no_piloto[audit_cols].copy()

# Columnas nuevas para completar después de la revisión visual
auditoria_aceptados_no_piloto["decision_auditoria"] = ""
auditoria_aceptados_no_piloto["motivo_auditoria"] = ""
auditoria_aceptados_no_piloto["observacion_auditoria"] = ""

# Sugerencia automática inicial, no definitiva
def sugerir_decision(row):
    cat = row["categoria_revision_v2"]

    if cat == "candidato_corpus_principal":
        return "revisar_para_corpus_principal"

    if cat == "revision_prenda_iconografica_prioritaria":
        return "revisar_para_corpus_principal_o_secundario"

    if cat == "revision_formato_longitudinal":
        return "revisar_para_secundario_o_principal"

    if cat == "revision_prenda_accesorio":
        return "revisar_para_secundario"

    if cat == "secundario_featherwork":
        return "pasar_a_corpus_secundario"

    if cat == "secundario_artefacto_textil":
        return "pasar_a_corpus_secundario"

    if cat == "descartar_material_sin_superficie":
        return "descartar"

    if cat == "descartar_implemento_textil":
        return "descartar"

    return "revisar"

auditoria_aceptados_no_piloto["decision_sugerida"] = auditoria_aceptados_no_piloto.apply(
    sugerir_decision,
    axis=1
)

output_csv = ROOT / "outputs/review/auditoria_aceptados_no_piloto.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)

auditoria_aceptados_no_piloto.to_csv(output_csv, index=False, encoding="utf-8-sig")

print("Archivo generado:")
print(output_csv)

display(
    auditoria_aceptados_no_piloto[
        [
            "id_objeto",
            "titulo_original",
            "cultura",
            "nombre_objeto_original",
            "categoria_revision_v2",
            "decision_sugerida",
            "decision_auditoria",
            "motivo_auditoria",
        ]
    ].head(20)
)

Archivo generado:
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_aceptados_no_piloto.csv


,id_objeto,titulo_original,cultura,nombre_objeto_original,categoria_revision_v2,decision_sugerida,decision_auditoria,motivo_auditoria
11,316983,Four-Cornered Hat,Tiwanaku,Hat,revision_prenda_iconografica_prioritaria,revisar_para_corpus_principal_o_secundario,,
12,316981,Four-Cornered Hat,Tiwanaku,Hat,revision_prenda_iconografica_prioritaria,revisar_para_corpus_principal_o_secundario,,
26,315772,Headband with ornamental tassels,Nasca,Tassels,revision_formato_longitudinal,revisar_para_secundario_o_principal,,
28,313148,Weaving Basket,Chancay,Basket with weaving implements,descartar_implemento_textil,descartar,,
29,314894,Square Cloth with Feathered Border,Chimú,Panel,secundario_featherwork,pasar_a_corpus_secundario,,
33,308078,"Border fragment, figures with staves",Paracas,Fragment,candidato_corpus_principal,revisar_para_corpus_principal,,
35,307944,Sash,Chimú,Sash,revision_formato_longitudinal,revisar_para_secundario_o_principal,,
40,308131,Skein of Camelid Hair,Peruvian,Skein,descartar_material_sin_superficie,descartar,,
41,316948,Openweave Cap,Peruvian,Hat,revision_prenda_accesorio,revisar_para_secundario,,
42,316910,Sash Fragment,Peruvian,Sash fragment,candidato_corpus_principal,revisar_para_corpus_principal,,


In [17]:
# ============================================================
# Resumen de decisiones sugeridas
# ============================================================

display(
    auditoria_aceptados_no_piloto["decision_sugerida"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "decision_sugerida", "decision_sugerida": "conteo"})
)

,conteo,count
0,revisar_para_corpus_principal,94
1,revisar_para_corpus_principal_o_secundario,22
2,revisar_para_secundario_o_principal,16
3,descartar,12
4,pasar_a_corpus_secundario,10
5,revisar_para_secundario,7


In [26]:
# ============================================================
# Exportar auditoría editable
# ============================================================

audit_path = ROOT / "outputs/review/auditoria_aceptados_no_piloto2.xlsx"

cols_export = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "url_objeto",
    "url_imagen",
    "categoria_revision_v2",
    "motivo_revision_v2",
    "decision_sugerida",
    "decision_auditoria",
    "motivo_auditoria",
    "observacion_auditoria",
]

cols_export = [c for c in cols_export if c in auditoria_aceptados_no_piloto.columns]

auditoria_aceptados_no_piloto[cols_export].to_excel(audit_path, index=False)

print(audit_path)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_aceptados_no_piloto2.xlsx


In [25]:
# ============================================================
# Exportar auditoría editable con links activos
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font

audit_path = ROOT / "outputs/review/auditoria_aceptados_no_piloto.xlsx"

cols_export = [
    "id_objeto",
    "titulo_original",
    "cultura",
    "periodo",
    "clasificacion_original",
    "nombre_objeto_original",
    "material_original",
    "url_objeto",
    "url_imagen",
    "categoria_revision_v2",
    "motivo_revision_v2",
    "decision_sugerida",
    "decision_auditoria",
    "motivo_auditoria",
    "observacion_auditoria",
]

cols_export = [c for c in cols_export if c in auditoria_aceptados_no_piloto.columns]

auditoria_aceptados_no_piloto[cols_export].to_excel(audit_path, index=False)

# Reabrir el Excel para convertir columnas I y J en hipervínculos
wb = load_workbook(audit_path)
ws = wb.active

for col in ["H", "I"]:
    for row in range(2, ws.max_row + 1):  # desde fila 2 para no tocar encabezados
        cell = ws[f"{col}{row}"]

        if cell.value and str(cell.value).startswith("http"):
            cell.hyperlink = cell.value
            cell.font = Font(color="0000FF", underline="single")

wb.save(audit_path)

print(audit_path)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_aceptados_no_piloto.xlsx


In [27]:
# ============================================================
# Validar decisiones y motivos insertados
# ============================================================

audit_path = ROOT / "outputs/review/auditoria_aceptados_no_piloto.xlsx"
audit = pd.read_excel(audit_path)

DECISIONES_VALIDAS = {
    "pasar_a_corpus_principal",
    "pasar_a_corpus_secundario",
    "descartar",
    "requiere_revision",
}

MOTIVOS_VALIDOS = {
    "superficie_textil_util",
    "fragmento_textil_util",
    "iconografia_legible",
    "composicion_visual_util",
    "iconografia_wari_tiwanaku_relevante",
    "patron_geometrico_legible",
    "prenda_tridimensional_util",
    "prenda_tridimensional_no_prioritaria",
    "formato_longitudinal_con_motivos",
    "formato_longitudinal_sin_motivos",
    "faja_o_banda_util_secundario",
    "featherwork_secundario",
    "artefacto_textil_secundario",
    "material_sin_superficie",
    "implemento_textil_no_analizable",
    "imagen_no_util",
    "fragmento_no_legible",
    "metadata_insuficiente",
}

audit["decision_auditoria"] = audit["decision_auditoria"].fillna("").str.strip()
audit["motivo_auditoria"] = audit["motivo_auditoria"].fillna("").str.strip()

errores_decision = audit[
    (audit["decision_auditoria"] != "") &
    (~audit["decision_auditoria"].isin(DECISIONES_VALIDAS))
]

errores_motivo = audit[
    (audit["motivo_auditoria"] != "") &
    (~audit["motivo_auditoria"].isin(MOTIVOS_VALIDOS))
]

pendientes = audit[
    (audit["decision_auditoria"] == "") |
    (audit["motivo_auditoria"] == "")
]

print("Total registros:", len(audit))
print("Completos:", len(audit) - len(pendientes))
print("Pendientes:", len(pendientes))
print("Errores en decisión:", len(errores_decision))
print("Errores en motivo:", len(errores_motivo))

display(errores_decision)
display(errores_motivo)
display(pendientes.head(20))

Total registros: 161
Completos: 27
Pendientes: 134
Errores en decisión: 0
Errores en motivo: 0


,id_objeto,titulo_original,cultura,periodo,clasificacion_original,nombre_objeto_original,material_original,url_objeto,url_imagen,categoria_revision_v2,motivo_revision_v2,decision_sugerida,decision_auditoria,motivo_auditoria,observacion_auditoria


,id_objeto,titulo_original,cultura,periodo,clasificacion_original,nombre_objeto_original,material_original,url_objeto,url_imagen,categoria_revision_v2,motivo_revision_v2,decision_sugerida,decision_auditoria,motivo_auditoria,observacion_auditoria


,id_objeto,titulo_original,cultura,periodo,clasificacion_original,nombre_objeto_original,material_original,url_objeto,url_imagen,categoria_revision_v2,motivo_revision_v2,decision_sugerida,decision_auditoria,motivo_auditoria,observacion_auditoria
0,313148,Weaving Basket,Chancay,NaN,Textiles-Implements,Basket with weaving implements,"Cane, shell, bone, fiber, camelid hair, wood, ...",https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_implemento_textil,"implemento asociado al tejido, no pieza textil...",descartar,descartar,,NaN
1,308131,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
2,308125,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
3,308126,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
4,308127,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
5,308128,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
6,308124,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
7,308130,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
8,308133,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN
9,308132,Skein of Camelid Hair,Peruvian,NaN,Textiles,Skein,Camelid hair,https://www.metmuseum.org/art/collection/searc...,https://images.metmuseum.org/CRDImages/ao/orig...,descartar_material_sin_superficie,"madeja/hilo: material textil, pero sin superfi...",descartar,descartar,,NaN


In [28]:
# ============================================================
# Guardar auditoría validada en CSV
# ============================================================

audit_csv = ROOT / "outputs/review/auditoria_aceptados_no_piloto_validada.csv"
audit.to_csv(audit_csv, index=False, encoding="utf-8-sig")

print(audit_csv)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_aceptados_no_piloto_validada.csv


In [29]:
# ============================================================
# Cargar auditoría completada y validar decision_auditoria
# ============================================================

audit_path = ROOT / "outputs/review/auditoria_aceptados_no_piloto.xlsx"
audit = pd.read_excel(audit_path)

audit["decision_auditoria"] = (
    audit["decision_auditoria"]
    .fillna("")
    .astype(str)
    .str.strip()
)

DECISIONES_VALIDAS = {
    "pasar_a_corpus_principal",
    "pasar_a_corpus_secundario",
    "descartar",
    "requiere_revision",
}

errores_decision = audit[
    ~audit["decision_auditoria"].isin(DECISIONES_VALIDAS)
]

pendientes_decision = audit[
    audit["decision_auditoria"] == ""
]

print("Total registros auditados:", len(audit))
print("Decisiones válidas:", len(audit) - len(errores_decision))
print("Decisiones con error:", len(errores_decision))
print("Pendientes:", len(pendientes_decision))

print("\nDistribución de decision_auditoria:")
display(
    audit["decision_auditoria"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "decision_auditoria", "decision_auditoria": "conteo"})
)

display(errores_decision)

Total registros auditados: 161
Decisiones válidas: 161
Decisiones con error: 0
Pendientes: 0

Distribución de decision_auditoria:


,conteo,count
0,pasar_a_corpus_principal,97
1,pasar_a_corpus_secundario,35
2,descartar,29


,id_objeto,titulo_original,cultura,periodo,clasificacion_original,nombre_objeto_original,material_original,url_objeto,url_imagen,categoria_revision_v2,motivo_revision_v2,decision_sugerida,decision_auditoria,motivo_auditoria,observacion_auditoria


In [30]:
# ============================================================
# Completar motivo_auditoria sugerido según decisión y categoría
# ============================================================

def sugerir_motivo_auditoria(row):
    decision = row.get("decision_auditoria", "")
    categoria = row.get("categoria_revision_v2", "")
    titulo = str(row.get("titulo_original", "")).lower()
    nombre = str(row.get("nombre_objeto_original", "")).lower()

    if decision == "descartar":
        if categoria == "descartar_material_sin_superficie":
            return "material_sin_superficie"
        if categoria == "descartar_implemento_textil":
            return "implemento_textil_no_analizable"
        return "imagen_no_util"

    if decision == "pasar_a_corpus_secundario":
        if categoria == "secundario_featherwork":
            return "featherwork_secundario"
        if categoria == "secundario_artefacto_textil":
            return "artefacto_textil_secundario"
        if categoria in ["revision_prenda_iconografica_prioritaria", "revision_prenda_accesorio"]:
            return "prenda_tridimensional_util"
        if categoria == "revision_formato_longitudinal":
            return "faja_o_banda_util_secundario"
        return "objeto_textil_secundario"

    if decision == "pasar_a_corpus_principal":
        if "tunic" in titulo or "tunic" in nombre:
            return "superficie_textil_util"
        if "mantle" in titulo or "mantle" in nombre:
            return "superficie_textil_util"
        if "bag" in titulo or "bag" in nombre:
            return "superficie_textil_util"
        if "panel" in titulo or "panel" in nombre:
            return "superficie_textil_util"
        if "fragment" in titulo or "fragment" in nombre:
            return "fragmento_textil_util"
        if "border" in titulo or "border" in nombre:
            return "iconografia_legible"
        return "composicion_visual_util"

    if decision == "requiere_revision":
        return "metadata_insuficiente"

    return ""

# Solo completa motivo_auditoria si está vacío
if "motivo_auditoria" not in audit.columns:
    audit["motivo_auditoria"] = ""

audit["motivo_auditoria"] = audit["motivo_auditoria"].fillna("").astype(str).str.strip()

audit.loc[
    audit["motivo_auditoria"] == "",
    "motivo_auditoria"
] = audit.loc[
    audit["motivo_auditoria"] == ""
].apply(sugerir_motivo_auditoria, axis=1)

display(
    audit[
        [
            "id_objeto",
            "titulo_original",
            "categoria_revision_v2",
            "decision_auditoria",
            "motivo_auditoria",
        ]
    ].head(30)
)

,id_objeto,titulo_original,categoria_revision_v2,decision_auditoria,motivo_auditoria
0,313148,Weaving Basket,descartar_implemento_textil,descartar,implemento_textil_no_analizable
1,308131,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
2,308125,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
3,308126,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
4,308127,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
5,308128,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
6,308124,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
7,308130,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
8,308133,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie
9,308132,Skein of Camelid Hair,descartar_material_sin_superficie,descartar,material_sin_superficie


In [31]:
# ============================================================
# Validar motivos de auditoría
# ============================================================

MOTIVOS_VALIDOS = {
    "superficie_textil_util",
    "fragmento_textil_util",
    "iconografia_legible",
    "composicion_visual_util",
    "iconografia_wari_tiwanaku_relevante",
    "patron_geometrico_legible",
    "prenda_tridimensional_util",
    "prenda_tridimensional_no_prioritaria",
    "formato_longitudinal_con_motivos",
    "formato_longitudinal_sin_motivos",
    "faja_o_banda_util_secundario",
    "featherwork_secundario",
    "artefacto_textil_secundario",
    "objeto_textil_secundario",
    "material_sin_superficie",
    "implemento_textil_no_analizable",
    "imagen_no_util",
    "fragmento_no_legible",
    "metadata_insuficiente",
}

errores_motivo = audit[
    ~audit["motivo_auditoria"].isin(MOTIVOS_VALIDOS)
]

print("Total registros:", len(audit))
print("Motivos válidos:", len(audit) - len(errores_motivo))
print("Motivos con error:", len(errores_motivo))

print("\nDistribución de motivo_auditoria:")
display(
    audit["motivo_auditoria"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "motivo_auditoria", "motivo_auditoria": "conteo"})
)

display(errores_motivo)

Total registros: 161
Motivos válidos: 161
Motivos con error: 0

Distribución de motivo_auditoria:


,conteo,count
0,superficie_textil_util,47
1,fragmento_textil_util,38
2,prenda_tridimensional_util,24
3,fragmento_no_legible,17
4,composicion_visual_util,12
5,material_sin_superficie,11
6,faja_o_banda_util_secundario,9
7,patron_geometrico_legible,2
8,implemento_textil_no_analizable,1


,id_objeto,titulo_original,cultura,periodo,clasificacion_original,nombre_objeto_original,material_original,url_objeto,url_imagen,categoria_revision_v2,motivo_revision_v2,decision_sugerida,decision_auditoria,motivo_auditoria,observacion_auditoria


In [32]:
# ============================================================
# Guardar auditoría con decisiones y motivos completados
# ============================================================

audit_csv = ROOT / "outputs/review/auditoria_aceptados_no_piloto_validada.csv"
audit_xlsx = ROOT / "outputs/review/auditoria_aceptados_no_piloto_validada.xlsx"

audit.to_csv(audit_csv, index=False, encoding="utf-8-sig")
audit.to_excel(audit_xlsx, index=False)

print("CSV:", audit_csv)
print("Excel:", audit_xlsx)

CSV: d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_aceptados_no_piloto_validada.csv
Excel: d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\auditoria_aceptados_no_piloto_validada.xlsx


In [33]:
# ============================================================
# Cargar auditoría validada
# ============================================================

from pathlib import Path
import pandas as pd

audit_csv = ROOT / "outputs/review/auditoria_aceptados_no_piloto_validada.csv"
audit_xlsx = ROOT / "outputs/review/auditoria_aceptados_no_piloto_validada.xlsx"

if audit_csv.exists():
    audit = pd.read_csv(audit_csv)
elif audit_xlsx.exists():
    audit = pd.read_excel(audit_xlsx)
else:
    audit = pd.read_excel(ROOT / "outputs/review/auditoria_aceptados_no_piloto.xlsx")

audit["decision_auditoria"] = audit["decision_auditoria"].fillna("").astype(str).str.strip()
audit["motivo_auditoria"] = audit["motivo_auditoria"].fillna("").astype(str).str.strip()

print("Registros auditados:", len(audit))

display(
    audit["decision_auditoria"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "decision_auditoria", "decision_auditoria": "conteo"})
)

Registros auditados: 161


,conteo,count
0,pasar_a_corpus_principal,97
1,pasar_a_corpus_secundario,35
2,descartar,29


In [34]:
# ============================================================
# Separar registros auditados según decisión final
# ============================================================

audit_principal = audit[audit["decision_auditoria"] == "pasar_a_corpus_principal"].copy()
audit_secundario = audit[audit["decision_auditoria"] == "pasar_a_corpus_secundario"].copy()
audit_descartado = audit[audit["decision_auditoria"] == "descartar"].copy()
audit_revision = audit[audit["decision_auditoria"] == "requiere_revision"].copy()

print("Pasan a corpus principal:", len(audit_principal))
print("Pasan a corpus secundario:", len(audit_secundario))
print("Descartados:", len(audit_descartado))
print("Requieren revisión:", len(audit_revision))

Pasan a corpus principal: 97
Pasan a corpus secundario: 35
Descartados: 29
Requieren revisión: 0


In [35]:
# ============================================================
# Cargar corpus actuales
# ============================================================

corpus_clean_actual = pd.read_csv(ROOT / "data/metadata/corpus_piloto_clean.csv")
corpus_secondary_actual = pd.read_csv(ROOT / "data/metadata/corpus_piloto_secondary.csv")

print("Corpus limpio actual:", len(corpus_clean_actual))
print("Corpus secundario actual:", len(corpus_secondary_actual))

Corpus limpio actual: 35
Corpus secundario actual: 15


In [36]:
# ============================================================
# Preparar registros auditados para integrarlos al corpus
# ============================================================

def preparar_auditados(df, destino):
    result = df.copy()

    result["origen_curacion"] = "auditoria_aceptados_no_piloto"
    result["decision_curacion_final"] = result["decision_auditoria"]
    result["motivo_curacion_final"] = result["motivo_auditoria"]
    result["destino_curacion_final"] = destino

    if "observacion_auditoria" not in result.columns:
        result["observacion_auditoria"] = ""

    return result

audit_principal_ready = preparar_auditados(audit_principal, "corpus_principal")
audit_secundario_ready = preparar_auditados(audit_secundario, "corpus_secundario")
audit_descartado_ready = preparar_auditados(audit_descartado, "descartado")
audit_revision_ready = preparar_auditados(audit_revision, "requiere_revision")

In [37]:
# ============================================================
# Unificar columnas entre corpus actual y registros auditados
# ============================================================

def unir_columnas(df1, df2):
    all_cols = list(dict.fromkeys(list(df1.columns) + list(df2.columns)))

    df1_aligned = df1.copy()
    df2_aligned = df2.copy()

    for col in all_cols:
        if col not in df1_aligned.columns:
            df1_aligned[col] = ""
        if col not in df2_aligned.columns:
            df2_aligned[col] = ""

    return df1_aligned[all_cols], df2_aligned[all_cols]

clean_actual_ready = corpus_clean_actual.copy()
clean_actual_ready["origen_curacion"] = "corpus_piloto_clean_actual"
clean_actual_ready["destino_curacion_final"] = "corpus_principal"

secondary_actual_ready = corpus_secondary_actual.copy()
secondary_actual_ready["origen_curacion"] = "corpus_piloto_secondary_actual"
secondary_actual_ready["destino_curacion_final"] = "corpus_secundario"

clean_actual_ready, audit_principal_ready = unir_columnas(
    clean_actual_ready,
    audit_principal_ready
)

secondary_actual_ready, audit_secundario_ready = unir_columnas(
    secondary_actual_ready,
    audit_secundario_ready
)

In [38]:
# ============================================================
# Construir corpus corregidos v2
# ============================================================

corpus_principal_v2 = pd.concat(
    [clean_actual_ready, audit_principal_ready],
    ignore_index=True
)

corpus_secondary_v2 = pd.concat(
    [secondary_actual_ready, audit_secundario_ready],
    ignore_index=True
)

corpus_descartados_auditoria = pd.concat(
    [audit_descartado_ready, audit_revision_ready],
    ignore_index=True
)

# Evitar duplicados por id_objeto
corpus_principal_v2 = corpus_principal_v2.drop_duplicates(subset=["id_objeto"], keep="first")
corpus_secondary_v2 = corpus_secondary_v2.drop_duplicates(subset=["id_objeto"], keep="first")
corpus_descartados_auditoria = corpus_descartados_auditoria.drop_duplicates(subset=["id_objeto"], keep="first")

print("Corpus principal v2:", len(corpus_principal_v2))
print("Corpus secundario v2:", len(corpus_secondary_v2))
print("Descartados/revisión auditoría:", len(corpus_descartados_auditoria))

Corpus principal v2: 132
Corpus secundario v2: 50
Descartados/revisión auditoría: 29


In [39]:
# ============================================================
# Validar cruces entre corpus principal y secundario
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x

ids_principal = set(corpus_principal_v2["id_objeto"].apply(norm_id))
ids_secundario = set(corpus_secondary_v2["id_objeto"].apply(norm_id))
ids_descartados = set(corpus_descartados_auditoria["id_objeto"].apply(norm_id))

cruce_principal_secundario = ids_principal & ids_secundario
cruce_principal_descartados = ids_principal & ids_descartados
cruce_secundario_descartados = ids_secundario & ids_descartados

print("Cruce principal/secundario:", len(cruce_principal_secundario))
print("Cruce principal/descartados:", len(cruce_principal_descartados))
print("Cruce secundario/descartados:", len(cruce_secundario_descartados))

print("IDs cruzados principal/secundario:", cruce_principal_secundario)

Cruce principal/secundario: 0
Cruce principal/descartados: 0
Cruce secundario/descartados: 0
IDs cruzados principal/secundario: set()


In [40]:
# ============================================================
# Guardar corpus corregidos v2
# ============================================================

out_dir = ROOT / "data/metadata"

path_principal_v2 = out_dir / "corpus_piloto_clean_v2.csv"
path_secondary_v2 = out_dir / "corpus_piloto_secondary_v2.csv"
path_descartados = out_dir / "corpus_piloto_descartados_auditoria.csv"

corpus_principal_v2.to_csv(path_principal_v2, index=False, encoding="utf-8-sig")
corpus_secondary_v2.to_csv(path_secondary_v2, index=False, encoding="utf-8-sig")
corpus_descartados_auditoria.to_csv(path_descartados, index=False, encoding="utf-8-sig")

print(path_principal_v2)
print(path_secondary_v2)
print(path_descartados)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\data\metadata\corpus_piloto_clean_v2.csv
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\data\metadata\corpus_piloto_secondary_v2.csv
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\data\metadata\corpus_piloto_descartados_auditoria.csv


In [41]:
# ============================================================
# Resumen antes/después de auditoría
# ============================================================

resumen_auditoria = pd.DataFrame([
    {
        "etapa": "corpus_principal_actual",
        "registros": len(corpus_clean_actual),
    },
    {
        "etapa": "recuperados_para_principal",
        "registros": len(audit_principal),
    },
    {
        "etapa": "corpus_principal_v2",
        "registros": len(corpus_principal_v2),
    },
    {
        "etapa": "corpus_secundario_actual",
        "registros": len(corpus_secondary_actual),
    },
    {
        "etapa": "recuperados_para_secundario",
        "registros": len(audit_secundario),
    },
    {
        "etapa": "corpus_secundario_v2",
        "registros": len(corpus_secondary_v2),
    },
    {
        "etapa": "descartados_por_auditoria",
        "registros": len(audit_descartado),
    },
    {
        "etapa": "requieren_revision",
        "registros": len(audit_revision),
    },
])

display(resumen_auditoria)

resumen_path = ROOT / "outputs/reports/resumen_auditoria_filtros_met.csv"
resumen_path.parent.mkdir(parents=True, exist_ok=True)
resumen_auditoria.to_csv(resumen_path, index=False, encoding="utf-8-sig")

print(resumen_path)

,etapa,registros
0,corpus_principal_actual,35
1,recuperados_para_principal,97
2,corpus_principal_v2,132
3,corpus_secundario_actual,15
4,recuperados_para_secundario,35
5,corpus_secundario_v2,50
6,descartados_por_auditoria,29
7,requieren_revision,0


d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\reports\resumen_auditoria_filtros_met.csv


In [42]:
# ============================================================
# Mostrar resumen antes/después de auditoría
# ============================================================

resumen_auditoria = pd.DataFrame([
    {"etapa": "corpus_principal_actual", "registros": len(corpus_clean_actual)},
    {"etapa": "recuperados_para_principal", "registros": len(audit_principal)},
    {"etapa": "corpus_principal_v2", "registros": len(corpus_principal_v2)},
    {"etapa": "corpus_secundario_actual", "registros": len(corpus_secondary_actual)},
    {"etapa": "recuperados_para_secundario", "registros": len(audit_secundario)},
    {"etapa": "corpus_secundario_v2", "registros": len(corpus_secondary_v2)},
    {"etapa": "descartados_por_auditoria", "registros": len(audit_descartado)},
    {"etapa": "requieren_revision", "registros": len(audit_revision)},
])

display(resumen_auditoria)

resumen_path = ROOT / "outputs/reports/resumen_auditoria_filtros_met.csv"
resumen_path.parent.mkdir(parents=True, exist_ok=True)
resumen_auditoria.to_csv(resumen_path, index=False, encoding="utf-8-sig")

print(resumen_path)

,etapa,registros
0,corpus_principal_actual,35
1,recuperados_para_principal,97
2,corpus_principal_v2,132
3,corpus_secundario_actual,15
4,recuperados_para_secundario,35
5,corpus_secundario_v2,50
6,descartados_por_auditoria,29
7,requieren_revision,0


d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\reports\resumen_auditoria_filtros_met.csv


In [43]:
# ============================================================
# Validar cruces entre corpus principal, secundario y descartados
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x

ids_principal = set(corpus_principal_v2["id_objeto"].apply(norm_id))
ids_secundario = set(corpus_secondary_v2["id_objeto"].apply(norm_id))
ids_descartados = set(corpus_descartados_auditoria["id_objeto"].apply(norm_id))

cruce_principal_secundario = ids_principal & ids_secundario
cruce_principal_descartados = ids_principal & ids_descartados
cruce_secundario_descartados = ids_secundario & ids_descartados

print("Cruce principal/secundario:", len(cruce_principal_secundario))
print("Cruce principal/descartados:", len(cruce_principal_descartados))
print("Cruce secundario/descartados:", len(cruce_secundario_descartados))

if cruce_principal_secundario:
    print("IDs cruzados principal/secundario:", cruce_principal_secundario)

if cruce_principal_descartados:
    print("IDs cruzados principal/descartados:", cruce_principal_descartados)

if cruce_secundario_descartados:
    print("IDs cruzados secundario/descartados:", cruce_secundario_descartados)

Cruce principal/secundario: 0
Cruce principal/descartados: 0
Cruce secundario/descartados: 0


In [44]:
# ============================================================
# Guardar corpus corregidos v2
# ============================================================

out_dir = ROOT / "data/metadata"

path_principal_v2 = out_dir / "corpus_piloto_clean_v2.csv"
path_secondary_v2 = out_dir / "corpus_piloto_secondary_v2.csv"
path_descartados = out_dir / "corpus_piloto_descartados_auditoria.csv"

corpus_principal_v2.to_csv(path_principal_v2, index=False, encoding="utf-8-sig")
corpus_secondary_v2.to_csv(path_secondary_v2, index=False, encoding="utf-8-sig")
corpus_descartados_auditoria.to_csv(path_descartados, index=False, encoding="utf-8-sig")

print(path_principal_v2)
print(path_secondary_v2)
print(path_descartados)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\data\metadata\corpus_piloto_clean_v2.csv
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\data\metadata\corpus_piloto_secondary_v2.csv
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\data\metadata\corpus_piloto_descartados_auditoria.csv


In [45]:
# ============================================================
# Generar reporte metodológico de auditoría - versión corregida
# ============================================================

report_path = ROOT / "outputs/reports/reporte_auditoria_filtros_met.md"

n_corpus_clean_actual = len(corpus_clean_actual)
n_audit_principal = len(audit_principal)
n_corpus_principal_v2 = len(corpus_principal_v2)
n_corpus_secondary_actual = len(corpus_secondary_actual)
n_audit_secundario = len(audit_secundario)
n_corpus_secondary_v2 = len(corpus_secondary_v2)
n_audit_descartado = len(audit_descartado)
n_audit_revision = len(audit_revision)

n_cruce_principal_secundario = len(cruce_principal_secundario)
n_cruce_principal_descartados = len(cruce_principal_descartados)
n_cruce_secundario_descartados = len(cruce_secundario_descartados)

report = f"""# Reporte de auditoría de filtros - The Met

## Objetivo

Revisar visualmente los registros aceptados por el filtro textil amplio que no habían pasado al corpus piloto, con el fin de identificar falsos descartes y mejorar la trazabilidad de los motivos de curación.

## Hallazgo principal

El filtro textil amplio funcionó como primera etapa de selección, pero el filtro morfológico hacia el corpus piloto fue demasiado restrictivo.

## Resultados antes/después

| Etapa | Registros |
|---|---:|
| Corpus principal actual | {n_corpus_clean_actual} |
| Recuperados para corpus principal | {n_audit_principal} |
| Corpus principal v2 | {n_corpus_principal_v2} |
| Corpus secundario actual | {n_corpus_secondary_actual} |
| Recuperados para corpus secundario | {n_audit_secundario} |
| Corpus secundario v2 | {n_corpus_secondary_v2} |
| Descartados por auditoría | {n_audit_descartado} |
| Requieren revisión | {n_audit_revision} |

## Validación de consistencia

| Cruce evaluado | Registros cruzados |
|---|---:|
| Principal / Secundario | {n_cruce_principal_secundario} |
| Principal / Descartados | {n_cruce_principal_descartados} |
| Secundario / Descartados | {n_cruce_secundario_descartados} |

## Interpretación metodológica

La auditoría permitió recuperar registros textiles que habían sido excluidos por criterios morfológicos demasiado restrictivos. En particular, se recuperaron fragmentos textiles, mantos, túnicas, paneles, bolsas y piezas con iconografía legible.

El corpus principal queda reservado para piezas con superficie textil visualmente analizable. El corpus secundario conserva textiles relevantes pero con morfología, función o materialidad especial, como prendas tridimensionales, formatos longitudinales, artefactos textiles y featherwork.

## Archivos generados

- `data/metadata/corpus_piloto_clean_v2.csv`
- `data/metadata/corpus_piloto_secondary_v2.csv`
- `data/metadata/corpus_piloto_descartados_auditoria.csv`
- `outputs/review/auditoria_aceptados_no_piloto_validada.csv`
- `outputs/reports/resumen_auditoria_filtros_met.csv`

## Conclusión

La versión v2 mejora la cobertura del corpus The Met y conserva trazabilidad sobre cada decisión de curación. Esta versión debe usarse como base para la siguiente etapa de revisión visual, anotación manual y experimentos multimodales.
"""

report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report, encoding="utf-8")

print(report_path)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\reports\reporte_auditoria_filtros_met.md


In [46]:
# ============================================================
# Recuperar URLs de imagen faltantes desde bases originales
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def norm_id(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x

def is_missing(x):
    if pd.isna(x):
        return True
    x = str(x).strip()
    return x == "" or x.lower() in ["nan", "none", "null"]

# Cargar corpus v2
corpus_principal_v2 = pd.read_csv(ROOT / "data/metadata/corpus_piloto_clean_v2.csv")
corpus_secondary_v2 = pd.read_csv(ROOT / "data/metadata/corpus_piloto_secondary_v2.csv")
corpus_descartados = pd.read_csv(ROOT / "data/metadata/corpus_piloto_descartados_auditoria.csv")

# Cargar fuentes originales donde sí estaban las URLs
fuentes = []

for path in [
    ROOT / "data/processed/met_candidatos_normalizados.csv",
    ROOT / "data/processed/met_textiles_pilot_clean.csv",
    ROOT / "data/processed/met_textiles_rejected.csv",
    ROOT / "data/metadata/corpus_piloto.csv",
]:
    if path.exists():
        df = pd.read_csv(path)
        df["_fuente_url"] = path.name
        fuentes.append(df)

base_urls = pd.concat(fuentes, ignore_index=True)

# Normalizar ID
base_urls["_id_norm"] = base_urls["id_objeto"].apply(norm_id)

# Detectar columnas útiles
cols_posibles = [
    "id_objeto",
    "_id_norm",
    "url_imagen",
    "primaryImage",
    "primaryImageSmall",
    "url_objeto",
    "objectURL",
    "_fuente_url",
]

cols_posibles = [c for c in cols_posibles if c in base_urls.columns]
base_urls = base_urls[cols_posibles].copy()

# Consolidar una sola URL de imagen y una URL de objeto
def first_from_row(row, cols):
    for col in cols:
        if col in row.index and not is_missing(row[col]):
            return str(row[col]).strip()
    return ""

base_urls["_url_imagen_ref"] = base_urls.apply(
    lambda row: first_from_row(row, ["url_imagen", "primaryImageSmall", "primaryImage"]),
    axis=1
)

base_urls["_url_objeto_ref"] = base_urls.apply(
    lambda row: first_from_row(row, ["url_objeto", "objectURL"]),
    axis=1
)

base_urls = base_urls[
    ["_id_norm", "_url_imagen_ref", "_url_objeto_ref", "_fuente_url"]
].drop_duplicates(subset=["_id_norm"], keep="first")

print("IDs con referencia de URL:", len(base_urls))
print("IDs con imagen:", base_urls["_url_imagen_ref"].ne("").sum())

IDs con referencia de URL: 295
IDs con imagen: 290


In [47]:
# ============================================================
# Completar url_imagen y url_objeto en corpus v2
# ============================================================

def completar_urls(df):
    result = df.copy()

    result["_id_norm"] = result["id_objeto"].apply(norm_id)

    if "url_imagen" not in result.columns:
        result["url_imagen"] = ""

    if "url_objeto" not in result.columns:
        result["url_objeto"] = ""

    result = result.merge(base_urls, on="_id_norm", how="left")

    mask_img = result["url_imagen"].apply(is_missing) & result["_url_imagen_ref"].notna()
    result.loc[mask_img, "url_imagen"] = result.loc[mask_img, "_url_imagen_ref"]

    mask_obj = result["url_objeto"].apply(is_missing) & result["_url_objeto_ref"].notna()
    result.loc[mask_obj, "url_objeto"] = result.loc[mask_obj, "_url_objeto_ref"]

    result = result.drop(columns=["_id_norm", "_url_imagen_ref", "_url_objeto_ref", "_fuente_url"], errors="ignore")

    return result

corpus_principal_v2 = completar_urls(corpus_principal_v2)
corpus_secondary_v2 = completar_urls(corpus_secondary_v2)
corpus_descartados = completar_urls(corpus_descartados)

def count_non_empty(df, col):
    if col not in df.columns:
        return 0
    return df[col].fillna("").astype(str).str.strip().replace("nan", "").ne("").sum()

print("Principal v2 con imagen:", count_non_empty(corpus_principal_v2, "url_imagen"), "/", len(corpus_principal_v2))
print("Secundario v2 con imagen:", count_non_empty(corpus_secondary_v2, "url_imagen"), "/", len(corpus_secondary_v2))
print("Descartados con imagen:", count_non_empty(corpus_descartados, "url_imagen"), "/", len(corpus_descartados))

Principal v2 con imagen: 132 / 132
Secundario v2 con imagen: 50 / 50
Descartados con imagen: 29 / 29


In [48]:
# ============================================================
# Guardar corpus v2 con URLs recuperadas
# ============================================================

corpus_principal_v2.to_csv(ROOT / "data/metadata/corpus_piloto_clean_v2.csv", index=False, encoding="utf-8-sig")
corpus_secondary_v2.to_csv(ROOT / "data/metadata/corpus_piloto_secondary_v2.csv", index=False, encoding="utf-8-sig")
corpus_descartados.to_csv(ROOT / "data/metadata/corpus_piloto_descartados_auditoria.csv", index=False, encoding="utf-8-sig")

print("CSV v2 actualizados con URLs de imagen.")

CSV v2 actualizados con URLs de imagen.


In [53]:
# ============================================================
# Función para generar galerías HTML v2
# ============================================================

from pathlib import Path
import pandas as pd
import html

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "outputs/review"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def is_valid_value(value):
    if pd.isna(value):
        return False
    value = str(value).strip()
    if value == "":
        return False
    if value.lower() in ["nan", "none", "null"]:
        return False
    return True

def first_non_empty(row, columns):
    for col in columns:
        if col in row.index and is_valid_value(row[col]):
            return str(row[col]).strip()
    return ""

FIELD_MAP = {
    "id_objeto": ["id_objeto", "objectID", "object_id"],
    "titulo": ["titulo_original", "title", "objectName"],
    "cultura": ["cultura", "culture"],
    "periodo": ["periodo", "period"],
    "clasificacion": ["clasificacion_original", "classification"],
    "objeto": ["nombre_objeto_original", "objectName", "object_name"],
    "material": ["material_original", "medium", "materials"],
    "imagen": ["url_imagen", "primaryImageSmall", "primaryImage", "image_url", "url_imagen_principal"],
    "url_objeto": ["url_objeto", "objectURL", "url", "link"],
    "decision": ["decision_curacion_final", "decision_auditoria", "destino_curacion_final"],
    "motivo": ["motivo_curacion_final", "motivo_auditoria", "motivo_revision_v2", "motivo_superficie"],
}

def build_gallery_v2(df, output_path, title):
    cards = []

    for _, row in df.iterrows():
        img = first_non_empty(row, FIELD_MAP["imagen"])
        obj_url = first_non_empty(row, FIELD_MAP["url_objeto"])

        object_id = first_non_empty(row, FIELD_MAP["id_objeto"])
        titulo = first_non_empty(row, FIELD_MAP["titulo"])
        cultura = first_non_empty(row, FIELD_MAP["cultura"])
        periodo = first_non_empty(row, FIELD_MAP["periodo"])
        clasificacion = first_non_empty(row, FIELD_MAP["clasificacion"])
        objeto = first_non_empty(row, FIELD_MAP["objeto"])
        material = first_non_empty(row, FIELD_MAP["material"])
        decision = first_non_empty(row, FIELD_MAP["decision"])
        motivo = first_non_empty(row, FIELD_MAP["motivo"])

        img_html = (
            f'<img src="{html.escape(img)}" loading="lazy">'
            if img else
            '<div class="no-image">Sin imagen</div>'
        )

        link_html = (
            f'<p><a href="{html.escape(obj_url)}" target="_blank">Ver objeto en The Met</a></p>'
            if obj_url else ""
        )

        card = f"""
        <div class="card">
            {img_html}
            <div class="info">
                <h3>{html.escape(object_id)} - {html.escape(titulo)}</h3>
                <p><b>Cultura:</b> {html.escape(cultura)}</p>
                <p><b>Periodo:</b> {html.escape(periodo)}</p>
                <p><b>Clasificación:</b> {html.escape(clasificacion)}</p>
                <p><b>Objeto:</b> {html.escape(objeto)}</p>
                <p><b>Material:</b> {html.escape(material)}</p>
                <p><b>Decisión final:</b> {html.escape(decision)}</p>
                <p><b>Motivo final:</b> {html.escape(motivo)}</p>
                {link_html}
            </div>
        </div>
        """
        cards.append(card)

    html_text = f"""
    <!doctype html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>{html.escape(title)}</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 24px;
                background: #f6f6f6;
            }}
            h1 {{
                margin-bottom: 4px;
            }}
            .summary {{
                margin-bottom: 24px;
                color: #333;
            }}
            .grid {{
                display: grid;
                grid-template-columns: repeat(auto-fill, minmax(320px, 1fr));
                gap: 18px;
            }}
            .card {{
                background: white;
                border: 1px solid #ddd;
                border-radius: 8px;
                overflow: hidden;
            }}
            img {{
                width: 100%;
                height: 280px;
                object-fit: contain;
                background: #eee;
                border-bottom: 1px solid #ddd;
            }}
            .no-image {{
                height: 280px;
                display: flex;
                align-items: center;
                justify-content: center;
                background: #ddd;
                color: #555;
            }}
            .info {{
                padding: 12px;
                font-size: 14px;
            }}
            h3 {{
                font-size: 15px;
                margin-top: 0;
            }}
            p {{
                margin: 6px 0;
            }}
        </style>
    </head>
    <body>
        <h1>{html.escape(title)}</h1>
        <p class="summary">Total registros: {len(df)}</p>
        <div class="grid">
            {''.join(cards)}
        </div>
    </body>
    </html>
    """

    output_path.write_text(html_text, encoding="utf-8")
    return output_path

In [54]:
html_principal = build_gallery_v2(
    corpus_principal_v2,
    OUTPUT_DIR / "corpus_principal_v2_galeria_corregida.html",
    "Corpus principal v2 - The Met"
)

html_secundario = build_gallery_v2(
    corpus_secondary_v2,
    OUTPUT_DIR / "corpus_secundario_v2_galeria_corregida.html",
    "Corpus secundario v2 - The Met"
)

html_descartados = build_gallery_v2(
    corpus_descartados,
    OUTPUT_DIR / "corpus_descartados_auditoria_galeria_corregida.html",
    "Descartados por auditoría - The Met"
)

print(html_principal)
print(html_secundario)
print(html_descartados)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\corpus_principal_v2_galeria_corregida.html
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\corpus_secundario_v2_galeria_corregida.html
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\corpus_descartados_auditoria_galeria_corregida.html


In [55]:
html_principal = build_gallery_v2(
    corpus_principal_v2,
    OUTPUT_DIR / "corpus_principal_v2_galeria_corregida.html",
    "Corpus principal v2 - The Met"
)

html_secundario = build_gallery_v2(
    corpus_secondary_v2,
    OUTPUT_DIR / "corpus_secundario_v2_galeria_corregida.html",
    "Corpus secundario v2 - The Met"
)

html_descartados = build_gallery_v2(
    corpus_descartados,
    OUTPUT_DIR / "corpus_descartados_auditoria_galeria_corregida.html",
    "Descartados por auditoría - The Met"
)

print(html_principal)
print(html_secundario)
print(html_descartados)

d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\corpus_principal_v2_galeria_corregida.html
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\corpus_secundario_v2_galeria_corregida.html
d:\Documentos\000. MSC\00. Tesis-Textiles\uni-cc-base-multimodal-textiles-andinos\outputs\review\corpus_descartados_auditoria_galeria_corregida.html
